# 第2回 宿題（発展）: LoRA で回答形式を学習し、学習前後を比較する

架空の講座の用語集を題材に、小さな指示チューニング済みモデルを LoRA で短時間学習し、学習の前後で **何が変わり、何が変わらないか** を比べます。評価は **形式遵守率**（`answer` と `in_glossary` の JSON を守れたか）と **内容正解率** に分けます。

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認します。ライブラリは基本的に uv で環境に導入します。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                              "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
import transformers
import peft
import trl
import accelerate
import datasets
import torch
print({m.__name__: m.__version__ for m in [
      transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定

**やること**: モデル名、学習のエポック数（データを何周するか）などを 1 つの辞書 `CONFIG` にまとめます。この辞書はセクション 7 の提出用ファイルにそのまま記録されるので、「どの条件で測ったか」が後から分かります。

- `revision` を固定しているのは、いつ実行しても同じ重みを使うためです。学習前後の比較や、他の人との比較で条件が揃います。
- `num_train_epochs: 20` は、学習データ（14 件）を 20 周する設定です。1 周が 2 ステップなので合計 40 ステップで、T4 で学習が 4〜5 分に収まります。
- `gen: {"do_sample": False}` は貪欲生成です。学習前後で生成条件を揃え、揺れの影響を除きます。

`USE_DRIVE` を `True` にすると Google Drive をマウントし、学習した adapter と結果ファイルを Drive にも保存します。セッションが切れても結果を残したいときに使います。1 回のセッションで完結させるなら `False` のままで構いません。

**見るところ**: 設定の JSON が表示されます。

In [ ]:
from pathlib import Path
import os
import json
import time

CONFIG = {
    # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 20,  # 学習データを何周学習するか。
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 32,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 1536,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    # 結果ファイルの保存先（左のファイル一覧から見られる）
    "work_dir": "./work/",
    "drive_dir": "./work/",  # USE_DRIVE=True のときの保存先
    "source_dir": "./sources/",
    "with_system_prompt": False,  # 学習にシステムプロンプトを含めるか否か,
    "data_augmentation": True
}
WORK = Path(CONFIG["work_dir"])
WORK.mkdir(parents=True, exist_ok=True)
SOURCE = Path(CONFIG["source_dir"])
WITH_SYSTEM_PROMPT = CONFIG["with_system_prompt"]
DATA_AUGMENTATION = CONFIG["data_augmentation"]

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

In [ ]:
import mlflow

# MLflow: 「パラメータ組 → 指標」を 1 実行ごとに記録し、並べて比較する
# 追跡データは ./mlflow/mlflow.db（sqlite、自動作成）、成果物（adapter・eval ファイル）は ./mlruns/ に入る
# CONFIG の値（num_train_epochs, lr, lora_r など）を変えてこのノートブックを再実行すると、セクション 9 で複数の run を一覧できる
mlflow.set_tracking_uri("sqlite:///mlflow/mlflow.db")
mlflow.set_experiment("lec02-lora")
# 前回の実行が途中で止まって run が閉じられていない場合は、先に閉じる
if mlflow.active_run():
    mlflow.end_run()
# run 名: 比較しそうな 3 パラメータを入れる
mlflow.start_run(
    run_name=f"e{CONFIG['num_train_epochs']}-lr{CONFIG['lr']}-r{CONFIG['lora_r']}")
# CONFIG をパラメータとして記録する（dict・list は JSON 文字列で保存）
mlflow.log_params({k: (json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v)
                  for k, v in CONFIG.items()})

## 2. 用語集と教師データの生成

**やること**: glossary_sft.jsonl から、学習用（train）、検証用（valid）、評価用（test）のデータを作ります。

モデルへの入力は「FAQ ＋ 質問」、望ましい出力は `{"answer": <文字列>, "in_glossary": <真偽値>}` の JSON 1 行です。

- 用語集にある質問には、用語集の文で答え、`in_glossary: true` にする。
- 用語集に無い質問には、「用語集に記載のない用語なので、回答できません。"」と答え、`in_glossary: false` にする。

**データの分け方**

- `glossary_sft.jsonl` のデータは同じ用語に対する質問を言い回しを変えて二回ずつ繰り返しているので、単純に最初の一回を学習用とする
- 二回目の質疑を `test` を `valid` に振り分ける

In [ ]:
if False:
    train_jsonl = []
    rest_jsonl = []
    test_jsonl = []
    valid_jsonl = []
    with open(SOURCE / "glossary_sft.jsonl", "r", encoding="utf-8") as f:
        json_lines = f.readlines()
        for i, json_text in enumerate(json_lines):
            if i % 2 == 0:
                dic = json.loads(json_text.strip())
                dic["messages"][1]["content"] = json.dumps(
                    {"answer": dic["messages"][1]["content"], "in_glossary": True}, ensure_ascii=False)
                train_jsonl.append(json.dumps(dic, ensure_ascii=False))
            else:
                rest_jsonl.append(json_text)

    for i, json_text in enumerate(rest_jsonl):
        dic = json.loads(json_text.strip())
        dic["messages"][1]["content"] = json.dumps(
            {"answer": dic["messages"][1]["content"], "in_glossary": True}, ensure_ascii=False)
        if i % 4 == 0:
            valid_jsonl.append(json.dumps(dic, ensure_ascii=False))
        else:
            test_jsonl.append(json.dumps(dic, ensure_ascii=False))

    (SOURCE / "train_grounded.jsonl").write_text("\n".join(train_jsonl), encoding="utf-8")
    (SOURCE / "test_grounded.jsonl").write_text("\n".join(test_jsonl), encoding="utf-8")
    (SOURCE / "valid_grounded.jsonl").write_text("\n".join(valid_jsonl), encoding="utf-8")

## 3. モデルの読み込みと学習前評価

**やること**: 基盤モデルを量子化なし（32bit）で読み込み、test の質問に答えさせて採点します。

プロンプト（`SYSTEM`）には「必ず JSON 1 行で、`answer` と `in_glossary` を返す」と書いてあります。つまり **形式は指示済み** です。このモデルがその指示を守れるかどうかが、学習前の確認点です。

**採点のしかた**（セクション 5 でも同じ関数を使います）

- **形式**: JSON として読め、キーが `answer` と `in_glossary` のちょうど 2 つで、型が文字列と真偽値であること。
- **内容**: `in_glossary` が正解と一致し、かつ正解の語句（例: 「9 月 30 日」と「23:59」）が `answer` に含まれること。

内容の判定には `in_glossary` の一致が必要です。そのため、キーが欠けている出力は、文面が正しくても内容は × になります。文面そのものは、表で自分の目で確かめてください。

`Qwen/Qwen3.5-0.8B` は画像も扱えるモデルですが、`AutoModelForCausalLM` で読み込むとテキスト部分だけが使われます。

**見るところ**

- 1 つ目のセル: `重みのメモリ(GB): 3.01` のように表示されます。モデルのダウンロード（約 1.75GB）に 1 分ほどかかります。
- 2 つ目のセル: 1 行の要約（`format_rate`、`content_rate` など）と、21 問の出力の表。評価には 1 分前後かかります。学習前は、多くの出力で `in_glossary` が欠けているはずです。

**出ても問題ない警告**

- `You are sending unauthenticated requests to the HF Hub`: Hugging Face にログインせずにモデルを取りに行っている、という案内です。公開モデルはログインしなくてもダウンロードできます。ログインが要るのは次の場合で、モデルの大きさでは決まりません。
  - 利用規約への同意が必要なモデル（gated model。Llama や Gemma など）や、非公開のリポジトリを使うとき。
  - 自分のモデルやデータを Hub にアップロードするとき。
  - 短時間に大量のリクエストを送り、回数の上限に当たるとき。上限はログインしていないと IP アドレスごとに 5 分あたり 3,000 回のファイル取得（2025 年 9 月時点の公表値。[Hub の rate limits](https://huggingface.co/docs/hub/rate-limits)）で、ログインすると上がります。
  
  このノートブックのモデルは公開されていて同意も不要です。読み込みで送るリクエストは多くても数十回なので、上限には届きません。もし上限に当たって `429 Too Many Requests` が出ても、ライブラリが待ち時間を読み取って自動で再試行します。ログインが必要になったときは、Colab の左側の鍵アイコン（シークレット）に Hugging Face のトークンを `HF_TOKEN` という名前で登録し、このノートブックからのアクセスを許可すると、自動で使われます。
- `... is falling back to its reference PyTorch implementation ...`: このモデルの一部の層（線形注意）に対する高速な実装が入っていないので、遅いが正しい実装で動くという案内です。4 行ほど出ます。

**ここで読んでおくこと**: 表を見て、「どのキーが欠けているか」「文面は FAQ と合っているか」を確認します。この表はセクション 5 で学習後の出力と並べて見ます。出力の原文は `/content/qlora_lab/eval_before.json` にも保存されています（左のファイル一覧から開けます）。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(
    CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:",
      DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

with open(SOURCE / "known_words.txt", "r", encoding="utf-8") as f:
    vocabulary = f.read()

# 形式はプロンプトで指示している。モデルがこれを守れるかが学習前の確認点
SYSTEM = '''
あなたはAIエンジニアリング講座に関する用語の回答係です。必ず JSON 1 行で答えてください。

形式: {"answer": 文字列, "in_glossary": 真偽値}。

用語集の語彙にある質問に対してはその意味を回答して "answer" に設定し、"in_glossary" を true とします。
用語集の語彙にない質問に対しては answer に『用語集に記載のない用語なので、回答できません。"』と書き、"in_glossary" を false にします。

用語集の語彙：
''' + vocabulary

print(SYSTEM)


def build_messages(example):
    return [{"role": "system", "content": SYSTEM}, example["messages"][0]]


def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(
        example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 出力の原文を省略せずに表示する


def check_format(text: str):
    '''JSON として読め、必須キーと型が合うか。失敗理由も返す'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict):
        return False, "not_object", None
    if set(obj) - {"answer", "in_glossary"}:
        return False, "extra_keys", obj
    if "answer" not in obj or "in_glossary" not in obj:
        return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["in_glossary"], bool):
        return False, "wrong_type", obj
    return True, "", obj


def check_content(obj, gold_entry):
    '''内容の合否: in_glossary が正解と一致し、必須語句を answer に含む'''
    if obj is None:
        return False
    if obj.get("in_glossary") != gold_entry["in_glossary"]:
        return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])


def mark(ok: bool) -> str:
    return "○" if ok else "×"


def evaluate(model, label: str, test, gold):
    '''test の質問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # 形式が壊れていても JSON らしい部分が読めるなら、内容だけは別に採点する
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try:
                loose = json.loads(m.group(0)) if m else None
            except Exception:
                loose = None
        question = ex["messages"][0]["content"]
        ok_content = check_content(loose if isinstance(
            loose, dict) else None, gold[question])
        rows.append({"question": question, "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_in_glossary": gold[question]["in_glossary"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary,
                                                         "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

In [ ]:
train_set = []
with open(SOURCE / "train_grounded.jsonl", "r", encoding="utf-8") as f:
    for line in f.readlines():
        train_set.append(json.loads(line))

with open(SOURCE / "train_ungrounded.jsonl", "r", encoding="utf-8") as f:
    for line in f.readlines():
        train_set.append(json.loads(line))

if DATA_AUGMENTATION:
    with open(SOURCE / "train_ex_grounded.jsonl", "r", encoding="utf-8") as f:
        for line in f.readlines():
            train_set.append(json.loads(line))

test_set = []
with open(SOURCE / "test_grounded.jsonl", "r", encoding="utf-8") as f:
    for line in f.readlines():
        test_set.append(json.loads(line))

with open(SOURCE / "test_ungrounded.jsonl", "r", encoding="utf-8") as f:
    for line in f.readlines():
        test_set.append(json.loads(line))

for data in train_set:
    print(data)

for data in test_set:
    print(data)

gold = {}
with open(SOURCE / "gold_grounded.json", "r", encoding="utf-8") as f:
    gold = json.load(f)

with open(SOURCE / "gold_ungrounded.json", "r", encoding="utf-8") as f:
    gold |= json.load(f)

print(gold)

before_summary, before_rows = evaluate(model, "before", test_set, gold)
# 質問ごとの出力。「正解の needs_review」と見比べ、どのキーが欠けているか、文面は FAQ と合っているかを読む
display(pd.DataFrame([{"質問": r["question"], "正解の in_glossary": r["gold_in_glossary"], "学習前の出力": r["raw"],
                       "形式": mark(r["format_ok"]), "形式の失敗理由": r["format_reason"], "内容": mark(r["content_ok"])} for r in before_rows]))

# MLflow: 学習前の評価指標と出力ファイルを記録する
mlflow.log_metrics(
    {f"before/{k}": v for k, v in before_summary.items() if k != "label"})
mlflow.log_artifact(str(WORK / "eval_before.json"))

## 4. LoRA 学習

### 学習の実行

**やること**: train データを chat 形式（system / user / assistant）に直し、LoRA を付けて学習します。学習した adapter は `adapter/` に保存します。

In [ ]:
valid_set = []
with open(SOURCE / "valid_grounded.jsonl", "r", encoding="utf-8") as f:
    for line in f.readlines():
        valid_set.append(json.loads(line))

with open(SOURCE / "valid_ungrounded.jsonl", "r", encoding="utf-8") as f:
    for line in f.readlines():
        valid_set.append(json.loads(line))

for data in valid_set:
    print(data)

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset


def to_chat(example):
    # 学習データは「入力（system + user）」と「望ましい出力（assistant）」の対。評価と同じ build_messages を使う
    ret = {"messages": build_messages(example) + [example["messages"][1]]}
    print(ret)
    return ret


if WITH_SYSTEM_PROMPT:
    train_ds = Dataset.from_list([to_chat(e) for e in train_set])
    valid_ds = Dataset.from_list([to_chat(e) for e in valid_set])
else:
    train_ds = Dataset.from_list(train_set)
    valid_ds = Dataset.from_list(valid_set)

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds,
                     eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
trainer.train()  # 途中で止まった場合の再開は 4b を参照
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil
    shutil.copytree(ADAPTER_DIR, Path(
        CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
# MLflow: 損失カーブ（logging_steps ごとに 1 点ずつ）、学習時間、adapter を記録する
for h in trainer.state.log_history:
    if "loss" in h:
        mlflow.log_metrics({"train/loss": h["loss"]}, step=h["step"])
mlflow.log_metrics({"train/seconds": round(time.time()-t0, 1),
                   "train/steps": trainer.state.global_step})
mlflow.log_artifacts(str(ADAPTER_DIR))
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

### 4b. 学習が止まった・できなかった場合

**途中で止まった（ランタイムの切断など）**: チェックポイントは 10 ステップごとに `/content/qlora_lab/ckpt/` に保存されています。ファイルが残っていれば、セクション 0〜3 を実行し直してから、上の学習セルの `trainer.train()` を `trainer.train(resume_from_checkpoint=True)` に書き換えて実行すると、最後のチェックポイントから 40 ステップ（20 エポック）まで続きを学習します。ランタイムごと消えてファイルが無い場合は、学習をやり直すか、下の方法に切り替えます。

**学習できなかった（GPU が確保できない、時間内に終わらない）**: 講師が同じノートブックを T4 で実行した結果（学習後の出力とスコア）を使って、比較と考察を進めます。セクション 5 以降のセルは学習後のモデル `tuned` が無いと動かないので、講師の結果を見ながら説明を読んでください。提出するときは、セクション 7 の `executed_training_myself` を `False` にし、本人が実行できた範囲を明記します。

## 5. 学習後評価（学習前と同じ test・同じ生成設定）

**やること**: 学習後のモデル `tuned` で、学習前とまったく同じ 21 問・同じ生成設定で評価し、前後を並べます。変えたのは adapter の有無だけです。

**見るところ**

- 比較表: `format_rate`（形式遵守率）と `content_rate`（内容正解率）が学習前からどう変わったか。
- `退行`: 学習前は形式も内容も通っていたのに、学習後に落ちた質問です。空のリストが期待値です。
- 質問ごとの表: 残った失敗を読み、どの型かを見分けてください。
  - 用語集になしの質問（正解の `in_glossary` が `false`）に、別項目を当てはめて断定した。
  - 用語集にありの質問（正解が `true`）に、「回答できません」と答えた。

In [ ]:
after_summary, after_rows = evaluate(tuned, "after", test_set, gold)
compare = pd.DataFrame([before_summary, after_summary]).set_index(
    "label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# 学習前は形式も内容も通ったのに、学習後に落ちた質問（退行）
regressed = [b["question"] for b, a in zip(
    before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

# 質問ごとの出力原文。形式（JSON・必須キー・型）と内容（needs_review の一致と必須語句）の判定を自分の目で確かめる
detail = pd.DataFrame([{"質問": b["question"], "正解の in_glossary": b["gold_in_glossary"],
                        "学習前の出力": b["raw"], "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}", "形式の失敗理由": a["format_reason"], }
                       for b, a in zip(before_rows, after_rows)])
display(detail)

# MLflow: 学習後の評価指標と出力ファイルを記録する
mlflow.log_metrics(
    {f"after/{k}": v for k, v in after_summary.items() if k != "label"})
mlflow.log_artifact(str(WORK / "eval_after.json"))

## 6. 知識を問う質問（形式を学んでも知識は保証されない）

**やること**: 形式の実験とは別に、FAQ にも学習データにも無い事実を 3 問尋ねます（対照として FAQ にある質問を 1 問加えます）。望ましい応答は「確認が必要（`needs_review: true`）」です。

応答を 3 つに分類します。

| 分類 | 意味 |
| --- | --- |
| `format_error` | JSON の形式を守れていない（学習前に多い） |
| `needs_review` | 「記載がありません。確認が必要です」と返した（望ましい応答） |
| `asserted` | `needs_review: false` で言い切った。FAQ に無い質問に対してなら、根拠のない断定（創作）です |

**見るところ**: 学習前は形式違反になりやすく、学習後は「確認が必要」と「断定」の両方が出ることがあります。出力の原文も表に出すので、断定した場合に何を根拠にしたように見えるか（FAQ の別項目を流用しているか）を読んでください。「Fine-Tuning では知識は絶対に直らない」と先に結論にせず、今回のデータでの結果を報告します。

学習前の出力は、adapter を一時的に無効にして取っています（`tuned.disable_adapter()`）。

In [ ]:
# KNOWLEDGE_QUESTIONS = [
#     "この講座の最終課題の締切はいつですか",       # FAQ に無い（架空）
#     "担当教員の名前は何ですか",                     # FAQ に無い
#     "次回の講義のテーマは何ですか",                 # FAQ に無い
#     "課題の提出期限はいつですか",                   # FAQ にある（対照）
# ]
#
#
# def classify(raw):
#     ok, _, obj = check_format(raw)
#     if not ok or obj is None:
#         return "format_error"
#     return "needs_review" if obj["needs_review"] else "asserted"
#
#
# knowledge_rows = []
# for q in KNOWLEDGE_QUESTIONS:
#     ex = make_example(q, "", True)
#     # LoRA は基盤モデルの層に差し込まれるので、学習後は `model` で生成しても adapter 込みになる。学習前の出力は adapter を一時的に無効にして取る
#     with tuned.disable_adapter():
#         before_raw = generate(tuned, ex)
#     after_raw = generate(tuned, ex)
#     knowledge_rows.append({"質問": q, "FAQ にある": q in gold, "学習前の分類": classify(before_raw), "学習前の出力": before_raw,
#                            "学習後の分類": classify(after_raw), "学習後の出力": after_raw})
# display(pd.DataFrame(knowledge_rows))

## 7. 比較表と提出

**やること**: 設定、ライブラリの版、学習前後の要約、退行した質問を 1 つの `result.json` にまとめます。

**提出するもの**

- `result.json`、`eval_before.json`、`eval_after.json`（`/content/qlora_lab` にあります。左のファイル一覧で右クリック → ダウンロード。`USE_DRIVE = True` なら Drive にもコピーされます）
- セクション 5 の比較表と、残った失敗の説明（どの型の失敗か）
- セクション 6 の結果（確認要求に変わった応答、断定が残った応答）
- 本人が学習を実行した範囲。講師の結果を使った場合は `executed_training_myself` を `False` にしてください。

In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    # "knowledge": knowledge_rows,
    "executed_training_myself": True,   # 講師の結果を使った場合は False にする
}
(WORK / "result.json").write_text(json.dumps(result,
                                             ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k !=
      "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])
# MLflow: 提出用ファイルを記録し、run を閉じる
mlflow.log_artifact(str(WORK / "result.json"))
mlflow.end_run()

## 8. 発展: 4bit 量子化で読み込み、メモリと出力を比べる（任意）

**やること**: 同じモデルを 4bit（NF4）で読み込み、重みのメモリと、学習前の出力がどう変わるかを見ます。QLoRA は、この 4bit の基盤モデルに LoRA を付けて学習する方法です。ここでは学習はしません。

- bitsandbytes は CUDA の GPU が前提なので、GPU ランタイムでだけ動きます。導入に 1 分ほどかかります。
- 4bit の読み込みは `device_map="auto"` で GPU に配置します（bitsandbytes の読み込み方式に合わせています）。
- 4bit 側で学習まで試す場合は、`prepare_model_for_kbit_training(model_4bit)` を通してからセクション 4 と同じ手順で学習し、学習前後とも 4bit の基盤モデルで評価します。

**見るところ**

- 重みのメモリ: 32bit の約 3GB に対して 4bit は 1GB 未満になります。
- 比較表と質問ごとの表: 学習していないので、32bit の学習前（`before`）との差は量子化の影響です。良くなる方向に変わる質問もあれば、`needs_review` が `"false"`（文字列）になって型検査で落ちる、根拠なしの質問に「はい」「なし」と断定する、といった出力も見られます。方向が一定でないことを確かめてください。

In [ ]:
# %pip install -q "bitsandbytes==0.50.*"
# from transformers import BitsAndBytesConfig
#
# bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # 保持精度: 4bit（NF4）
#                          bnb_4bit_compute_dtype=torch.float16,             # 計算精度: 16bit（T4 は bf16 に対応していない）
#                          bnb_4bit_use_double_quant=True)
# model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
# print("重みのメモリ(GB) | 量子化なし（32bit）:", BASE_MEMORY_GB, "| 4bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))
#
# # 学習はしていないので、量子化なしの学習前（before）との差が量子化の影響
# q_summary, q_rows = evaluate(model_4bit, "before_4bit", test_set, gold)
# display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
# display(pd.DataFrame([{"質問": r["question"], "正解の in_glossary": r["gold_in_glossary"], "4bit・学習前の出力": r["raw"],
#                        "形式/内容": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}", "形式の失敗理由": r["format_reason"]} for r in q_rows]))

## 9. パラメータごとの結果比較（MLflow）

**やること**: `CONFIG` の値（`num_train_epochs`、`lr`、`lora_r` など）を変えてこのノートブックを再実行すると、1 実行ごとに run が記録されます（セクション 1 の run 名、例: `e20-lr0.0002-r16`）。追跡データは `./mlflow/mlflow.db`（sqlite）、成果物（adapter、eval ファイル）は `./mlruns/` にあります。このセルで各 run の指標を 1 つの表に並べます。

**見るところ**: `train/loss` 列は最後のステップの損失です。`num_train_epochs` を増やすと `after/content_rate` がどう動くか、`lr` を変えるとどうなるかなどを横に見比べてください。

**詳しく見るには**: `uv run mlflow ui --backend-store-uri "sqlite:///mlflow/mlflow.db" --default-artifact-root "./mlruns" --serve-artifacts` でブラウザが開き、損失カーブや成果物のダウンロードができます。

- Colab では `./mlflow/` と `./mlruns/` はセッション終了で消えるので、終わる前に Drive にコピーしてください（`%pip install -q mlflow` が必要）。
- セクション 8（4bit）は任意の発展なので、MLflow には記録されていません。

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
exp = client.get_experiment_by_name("lec02-lora")
runs = client.search_runs([exp.experiment_id], order_by=[
                          "attributes.start_time ASC"])
rows = []
for r in runs:
    row = {"run": r.info.run_name,
           "開始": time.strftime("%m-%d %H:%M", time.localtime(r.info.start_time/1000))}
    row.update({k: r.data.params[k] for k in ("num_train_epochs", "warmup_steps", "lr", "lora_r", "lora_alpha", "max_seq_len", "with_system_prompt", "data_augmentation")
                if k in r.data.params})
    # MLflow 3.x では run.data.metrics が「名前 → 最新の値」（train/loss は最後のステップの値）
    for m in ("train/loss", "train/seconds", "before/format_rate", "before/content_rate",
              "after/format_rate", "after/content_rate", "after/both_rate"):
        row[m] = r.data.metrics.get(m)
    rows.append(row)
display(pd.DataFrame(rows).set_index("run"))

# mlflow での確認

```
uv run mlflow ui --backend-store-uri "sqlite:///mlflow/mlflow.db" --default-artifact-root "./mlruns" --serve-artifacts
```